# Navigation — DQN agent for the Unity Banana environment

This notebook drives the agent end-to-end:

1. Connect to the Unity Banana environment (downloaded separately).
2. Inspect the action and state spaces.
3. Train a Double-DQN + Dueling Q-network agent until the rolling-100
   episode score crosses **+13** (the project's solve criterion).
4. Plot the learning curve.
5. Reload the saved weights and watch a trained episode.

The agent and Q-network live in `agent.py` and `model.py` next to this
notebook.

In [ ]:
import os
from collections import deque
from time import time

import numpy as np
import torch
import matplotlib.pyplot as plt

from unityagents import UnityEnvironment

from agent import DQNAgent

%matplotlib inline

## 1. Connect to the environment

Download the Banana env binary that matches your OS from the Udacity project page and point `file_name` to the executable.

In [ ]:
env = UnityEnvironment(file_name='Banana_Linux/Banana.x86_64')
brain_name = env.brain_names[0]
brain      = env.brains[brain_name]

env_info  = env.reset(train_mode=True)[brain_name]
state     = env_info.vector_observations[0]
state_size  = len(state)
action_size = brain.vector_action_space_size
print(f'state_size  = {state_size}')
print(f'action_size = {action_size}')

## 2. Train the agent

Hyper-parameters (all in `agent.py`):

| | |
|---|---|
| `buffer_size` | 100 000 |
| `batch_size`  | 64 |
| `gamma`       | 0.99 |
| `tau`         | 1e-3 |
| `lr`          | 5e-4 |
| `update_every`| 4 steps |
| `dueling`     | True |
| `double_dqn`  | True |
| ε start / min / decay | 1.0 / 0.01 / 0.995 |

In [ ]:
def dqn(n_episodes=2000, eps_start=1.0, eps_end=0.01, eps_decay=0.995,
        target_score=13.0):
    agent = DQNAgent(state_size=state_size, action_size=action_size)
    scores_window = deque(maxlen=100)
    all_scores = []
    eps = eps_start
    started = time()
    for episode in range(1, n_episodes + 1):
        env_info = env.reset(train_mode=True)[brain_name]
        state    = env_info.vector_observations[0]
        score    = 0.0
        while True:
            action = agent.act(state, eps)
            env_info = env.step(action)[brain_name]
            next_state = env_info.vector_observations[0]
            reward     = env_info.rewards[0]
            done       = env_info.local_done[0]
            agent.step(state, action, reward, next_state, int(done))
            state, score = next_state, score + reward
            if done:
                break
        eps = max(eps_end, eps_decay * eps)
        scores_window.append(score); all_scores.append(score)
        avg = np.mean(scores_window)
        print(f'\rEp {episode:4d}\tscore={score:.1f}\trolling100={avg:.2f}\teps={eps:.3f}', end='')
        if episode % 100 == 0:
            print(f'\n[Episode {episode:4d}]  rolling-100 mean = {avg:.2f}  ({time() - started:.0f}s elapsed)')
        if avg >= target_score and len(scores_window) == 100:
            print(f'\n\nSolved in {episode} episodes — saving banana_qnet.pth.')
            torch.save(agent.qnet_local.state_dict(), 'banana_qnet.pth')
            return agent, all_scores
    return agent, all_scores

agent, scores = dqn()

## 3. Plot the learning curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(scores, alpha=0.4, label='per-episode score')
window = 100
if len(scores) >= window:
    rolling = np.convolve(scores, np.ones(window) / window, mode='valid')
    ax.plot(np.arange(len(rolling)) + window, rolling, color='C1', lw=2, label='rolling 100')
ax.axhline(13, color='C2', linestyle='--', label='solve threshold (+13)')
ax.set_xlabel('episode'); ax.set_ylabel('score'); ax.legend()
plt.tight_layout(); plt.show()
np.save('banana_scores.npy', np.array(scores))

## 4. Watch the trained agent

In [ ]:
agent_eval = DQNAgent(state_size=state_size, action_size=action_size)
agent_eval.qnet_local.load_state_dict(torch.load('banana_qnet.pth', map_location='cpu'))

env_info = env.reset(train_mode=False)[brain_name]
state    = env_info.vector_observations[0]
score    = 0.0
while True:
    action = agent_eval.act(state, eps=0.0)
    env_info = env.step(action)[brain_name]
    state    = env_info.vector_observations[0]
    score   += env_info.rewards[0]
    if env_info.local_done[0]:
        break
print('Demo episode score:', score)

In [ ]:
env.close()

See **Report.md** for the full writeup of the algorithm, hyper-parameters, and ideas for future work.